In [2]:
#Import Library 

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    make_scorer,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

In [3]:
#Load dataset 
processed_dir = Path("D:\Erdos\Data Science Bootcamp\summer26-diabetes-risk\early_diabetes_screening\data\processed")
results_dir = Path(r"D:\Erdos\Data Science Bootcamp\summer26-diabetes-risk\early_diabetes_screening\results")
results_dir.mkdir(parents=True, exist_ok=True)
modeling_dir = results_dir / "modeling"
modeling_dir.mkdir(parents=True, exist_ok=True)
today = datetime.today().strftime("%Y_%m_%d")
X = pd.read_csv(processed_dir / "X_pred_engineered.csv")
y = pd.read_csv(processed_dir / "y_target_selected.csv").squeeze()
print("Predictor shape", X.shape)
print("Target shape", y.shape)
X.head()

Predictor shape (9232, 37)
Target shape (9232,)


,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,BMXWAIST,BPQ020,BPQ080,SMQ020,...,high_waist,physically_active,ever_regular_alcohol,smoking_history,fair_poor_diet,frequent_fast_food,age_bmi,age_waist,age_systolic_bp,metabolic_risk_score
0,29.0,2.0,6.0,5.0,5.00,37.8,117.9,2.0,1.0,2.0,...,1.0,1.0,1.0,0.0,0.0,1.0,1096.2,3419.1,2871.0,2.0
1,21.0,2.0,2.0,4.0,5.00,NaN,NaN,2.0,2.0,2.0,...,NaN,1.0,NaN,0.0,0.0,1.0,NaN,NaN,NaN,0.0
2,49.0,1.0,3.0,2.0,NaN,29.7,120.4,2.0,1.0,1.0,...,1.0,0.0,1.0,1.0,0.0,0.0,1455.3,5899.6,5243.0,2.0
3,36.0,1.0,3.0,4.0,0.83,21.9,86.8,2.0,2.0,1.0,...,0.0,1.0,1.0,1.0,1.0,0.0,788.4,3124.8,4092.0,2.0
4,68.0,1.0,7.0,4.0,1.20,30.2,109.6,1.0,1.0,2.0,...,1.0,1.0,1.0,0.0,0.0,0.0,2053.6,7452.8,9112.0,3.0


In [4]:
# Stratified final holdout split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

Training set shape: (7385, 37)
Test set shape: (1847, 37)

Training target distribution:
diabetes
0    0.8028
1    0.1972
Name: proportion, dtype: float64

Test target distribution:
diabetes
0    0.8029
1    0.1971
Name: proportion, dtype: float64


In [5]:
# Define categoric and numeric features 

coded_categorical_features = [
    # Original coded categorical variables
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "BPQ020",
    "BPQ080",
    "SMQ020",
    "PAQ650",
    "PAQ665",
    "ALQ111",
    "ALQ121",
    "DBQ700",
    
    # Engineered categorical / binary indicator variables
    "obese",
    "bmi_category",
    "high_bp_exam",
    "high_waist",
    "physically_active",
    "ever_regular_alcohol",
    "smoking_history",
    "fair_poor_diet",
    "fair_or_poor_diet",
    "frequent_fast_food"
]

categorical_features = [
    col for col in coded_categorical_features
    if col in X_train.columns
]

object_categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

categorical_features = sorted(list(set(categorical_features + object_categorical_features)))

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Number of numeric features:", len(numeric_features))
print(numeric_features)

print("\nNumber of categorical features:", len(categorical_features))
print(categorical_features)

Number of numeric features: 17
['RIDAGEYR', 'INDFMPIR', 'BMXBMI', 'BMXWAIST', 'PAD680', 'ALQ130', 'DBD895', 'DBD900', 'DBD905', 'DBD910', 'avg_systolic_bp', 'avg_diastolic_bp', 'pulse_pressure', 'age_bmi', 'age_waist', 'age_systolic_bp', 'metabolic_risk_score']

Number of categorical features: 20
['ALQ111', 'ALQ121', 'BPQ020', 'BPQ080', 'DBQ700', 'DMDEDUC2', 'PAQ650', 'PAQ665', 'RIAGENDR', 'RIDRETH3', 'SMQ020', 'bmi_category', 'ever_regular_alcohol', 'fair_poor_diet', 'frequent_fast_food', 'high_bp_exam', 'high_waist', 'obese', 'physically_active', 'smoking_history']


In [6]:
# Preprocessing pipeline 

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [7]:
# Define baseline models 
models = {
    "Dummy Classifier": DummyClassifier(
        strategy="most_frequent"
    ),
    
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42
    )
}

In [8]:
# Define metrics and cross-validations 

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": make_scorer(recall_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score)
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
# Cross validation on trainning set 

cv_results_list = []

for model_name, model in models.items():
    print(f"Running cross-validation for: {model_name}")
    
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    
    row = {
        "model": model_name,
        "split_strategy": "80_20_stratified_holdout_with_5fold_stratified_cv_on_training"
    }
    
    for metric in scoring.keys():
        scores = cv_results[f"test_{metric}"]
        row[f"{metric}_mean"] = scores.mean()
        row[f"{metric}_std"] = scores.std()
    
    cv_results_list.append(row)
    
cv_results_df = pd.DataFrame(cv_results_list)
cv_results_df

Running cross-validation for: Dummy Classifier
Running cross-validation for: Logistic Regression
Running cross-validation for: Decision Tree


,model,split_strategy,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std
0,Dummy Classifier,80_20_stratified_holdout_with_5fold_stratified...,0.802844,0.000271,0.500000,0.000000,0.500000,0.000000,0.197156,0.000271,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,Logistic Regression,80_20_stratified_holdout_with_5fold_stratified...,0.716723,0.003547,0.730046,0.006979,0.805430,0.008478,0.479989,0.011408,0.752048,0.016208,0.387462,0.004708,0.511398,0.007113
2,Decision Tree,80_20_stratified_holdout_with_5fold_stratified...,0.670819,0.018450,0.702781,0.008835,0.763454,0.010693,0.400736,0.017537,0.755531,0.033227,0.347103,0.011145,0.475211,0.009273


In [10]:
cv_results_rounded = cv_results_df.copy()

numeric_cols = cv_results_rounded.select_dtypes(include=["float64"]).columns
cv_results_rounded[numeric_cols] = cv_results_rounded[numeric_cols].round(4)

display_cols = [
    "model",
    "recall_mean",
    "pr_auc_mean",
    "roc_auc_mean",
    "balanced_accuracy_mean",
    "f1_mean",
    "precision_mean",
    "accuracy_mean"
]

cv_results_rounded[display_cols]

,model,recall_mean,pr_auc_mean,roc_auc_mean,balanced_accuracy_mean,f1_mean,precision_mean,accuracy_mean
0,Dummy Classifier,0.0000,0.1972,0.5000,0.5000,0.0000,0.0000,0.8028
1,Logistic Regression,0.7520,0.4800,0.8054,0.7300,0.5114,0.3875,0.7167
2,Decision Tree,0.7555,0.4007,0.7635,0.7028,0.4752,0.3471,0.6708


### Baseline Cross-Validation Results

Three baseline models were evaluated using stratified 5-fold cross-validation: Dummy Classifier, Logistic Regression, and Decision Tree. These models were used as simple reference models before moving to more complex modeling experiments.

The Dummy Classifier achieved high accuracy of 0.8028 because it predicts the majority class, non-diabetes. However, its recall, precision, and F1-score for the diabetes class were all 0. This shows that accuracy alone is misleading for this imbalanced screening problem.

Logistic Regression achieved a recall of 0.7520, PR-AUC of 0.4800, ROC-AUC of 0.8054, balanced accuracy of 0.7300, and F1-score of 0.5114. This indicates that Logistic Regression provides a strong and interpretable baseline for identifying diabetes cases.

The Decision Tree achieved a slightly higher recall of 0.7555, but its PR-AUC, ROC-AUC, balanced accuracy, F1-score, precision, and accuracy were lower than Logistic Regression. This suggests that although the Decision Tree identifies a similar proportion of diabetes cases, Logistic Regression provides better overall performance across most evaluation metrics.

Overall, Logistic Regression was selected as the strongest baseline model because it balances high recall with better PR-AUC, ROC-AUC, balanced accuracy, and interpretability. More complex models should be compared against this baseline and justified only if they provide clear improvements in the project KPIs.



In [11]:
# Save the baseline results 

cv_results_path = modeling_dir / f"baseline_cv_results.csv"
cv_results_rounded.to_csv(cv_results_path, index=False)